In [0]:
# %restart_python

In [0]:
%pip install "loguru==0.7.3"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:

from pathlib import Path

PROJECT_ROOT=Path("/Workspace/Users/dax.westerman@vumc.org/tedla-hypertension-project/v0.0.2")
LOGS_FOLDER = PROJECT_ROOT / "logs"
DEDUPED_FLAG_PATH = LOGS_FOLDER / "deduped_flag.txt"
SOURCE_MRN_CROSSWALK_FILE = "raw_data/patient_mrns/0.0.2_p6290_xwalk_mrn_deid.csv"

In [0]:
from loguru import logger
from .common.table_types import ProjectView
from .common.table_types import ProjectTable
from .common.table_definitions.project._corpus_features import CorpusFeaturesTable
from .common.table_definitions.project import (
    get_mrns_table,
    get_cohort_table,
    create_corpus_features_table,
    create_note_types_table,
    get_corpus_definition_view,
)

# Added logger for the notebook
logger.remove()
logger.add(lambda msg: print(msg, end=""))
logger.add(LOGS_FOLDER / "error.logs", level="ERROR")
logger.add(LOGS_FOLDER / "processing.log", level="DEBUG")

In [0]:
#####################################################################################
# Build MRNs Table
#
# Takes the raw CSV file containing the MRN crosswalk and the cleaned
# patient cohort parquet file to create the MRNs table in the project.
#####################################################################################

from common.files import CSVFileWrapper, ParquetFileWrapper
from common.table_types import RDTable
rd_note_table = RDTable("note")
rd_visit_occurrence_table = RDTable("visit_occurrence")
rd_person_table = RDTable("person")

mrns_table: ProjectTable

with CSVFileWrapper(
    name="patient_mrn_crosswalk",
    workspace_path=PROJECT_ROOT / SOURCE_MRN_CROSSWALK_FILE,
    dbfs_file_name="patient_cohort",
    table_name="mrns",
) as mrn_crosswalk_csv_file:

    with ParquetFileWrapper(
        name="patient_cohort", workspace_path=PROJECT_ROOT / "cleaned_data"
    ) as patient_cohort_parquet_file:
        patient_cohort_parquet_file.move_to("dbfs")

        mrns_table: ProjectTable = get_mrns_table(
            patient_cohort_parquet_file, mrn_crosswalk_csv_file
        )
        # Create the MRNs table
        mrns_table.create()
        
        

In [0]:
# Build Cohort Table
cohort_table: ProjectTable = get_cohort_table(mrns_table, rd_person_table)
cohort_table.create()

In [0]:
# Build Corpus Features Table
corpus_features_source: CorpusFeaturesTable = create_corpus_features_table(cohort_table, rd_note_table, rd_visit_occurrence_table)


In [0]:
# Build Note Types Table
note_types_table: ProjectTable = create_note_types_table()


In [0]:
# Build Corpus Definition View
corpus_definition_view: ProjectView = get_corpus_definition_view(cohort_table, rd_note_table, rd_visit_occurrence_table)
corpus_definition_view.create()

In [0]:

# ......................................................................
# Static definitions of note groups
# ......................................................................

from .common.table_definitions.project.note_groups import (
    AdmissionNotesTable,
    CommunicationEncounterNotesTable,
    DischargeSummaryNotesTable,
    ECGImpressionNotesTable,
    EmergencyDepartmentNotesTable,
    InpatientNotesTable,
    OutpatientNotesTable,
    ProblemListNotesTable,
)

admission_notes_table: AdmissionNotesTable = AdmissionNotesTable(corpus_features_source).create()
communication_encounter_notes_table: CommunicationEncounterNotesTable = CommunicationEncounterNotesTable(
    corpus_features_source
).create()
discharge_summary_notes_table: DischargeSummaryNotesTable = DischargeSummaryNotesTable(corpus_features_source).create()
ecg_impression_notes_table: ECGImpressionNotesTable = ECGImpressionNotesTable(corpus_features_source).create()
emergency_department_notes_table: EmergencyDepartmentNotesTable = EmergencyDepartmentNotesTable(corpus_features_source).create()
inpatient_notes_table: InpatientNotesTable = InpatientNotesTable(corpus_features_source).create()
outpatient_notes_table: OutpatientNotesTable = OutpatientNotesTable(corpus_features_source).create()
problem_list_notes_table: ProblemListNotesTable = ProblemListNotesTable(corpus_features_source).create()